# AI Music Generation Engine — Neural Network Trained on Classical Piano

**Client:** Confidential creative tech company 
**Role:** ML Engineer — MLP Architecture, Backpropagation from Scratch, Generative AI 
**Result:** Trained 2-layer MLP that generates original compositions in the style of Chopin

---

## Project Overview

A creative tech client needed a generative AI system that could learn musical patterns and compose original pieces in the style of classical composers.

I built a 2-layer MLP neural network **entirely from scratch in NumPy** — implementing the full forward pass, softmax output layer, and vectorized backpropagation. Trained on Chopin MIDI data with one-hot encoded note sequences, the model learns to predict the next note given a 20-note context window, generating new compositions one note at a time.

**Pipeline:**
1. MIDI parsing and note sequence extraction
2. Sliding window data generation (context length = 20)
3. One-hot encoding of note indices → 2,560-dimensional feature vectors
4. 2-layer MLP: input (2560) → hidden (100, ReLU) → output (128, softmax)
5. Vectorized backpropagation with mini-batch SGD
6. Autoregressive generation — sample next note, append to context, repeat

**Stack:** Python · NumPy · mido · matplotlib

## 1. Imports & Setup

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from mido import MidiFile, MidiTrack, Message

# Install mido if needed: pip install mido

CONTEXT_LENGTH = 20   # number of preceding notes used as input
NUM_NOTES      = 128  # MIDI note range (0–127)
RANDOM_STATE   = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

## 2. Data — MIDI Parsing

Each MIDI file is parsed into a sequence of note integers (0–127, where 60 = middle C). The training set consists of simplified Chopin piano pieces — note durations and velocities have been normalized so the model focuses purely on **pitch sequences**.

The simplification is intentional: it reduces the output space to 128 discrete classes, making the generation task tractable for an MLP.

In [ ]:
def get_midi_notes(filename):
    """
    Parse a MIDI file and return the sequence of note-on pitch values.

    Parameters
    ----------
    filename : str — path to .mid file

    Returns
    -------
    list of int — pitch values in [0, 127]
    """
    mid = MidiFile(filename)
    notes = []
    for track in mid.tracks:
        for msg in track:
            if msg.type == 'note_on' and msg.velocity > 0:
                notes.append(msg.note)
    return notes

In [ ]:
# Load training pieces
import glob

chopin_files = sorted(glob.glob('data/chopin/*_simplified.mid'))
print(f'Found {len(chopin_files)} Chopin pieces:')
for f in chopin_files:
    notes = get_midi_notes(f)
    print(f'  {f.split("/")[-1]:<45} {len(notes)} notes')

## 3. Sliding Window Data Generation

For each piece, we slide a window of `CONTEXT_LENGTH=20` notes across the sequence. Each window becomes one training sample: the 20 preceding notes are the input `x`, and the following note is the target `t`.

A piece with N notes produces `N - CONTEXT_LENGTH` training pairs.

In [ ]:
def gen_input_output(notes, context_length=CONTEXT_LENGTH):
    """
    Generate (x, t) training pairs using a sliding window over a note sequence.

    Parameters
    ----------
    notes          : list of int — full note sequence
    context_length : int — window size (number of input notes)

    Returns
    -------
    list of (x, t) where x is a list of `context_length` notes and t is the next note
    """
    return [
        (notes[i : i + context_length], notes[i + context_length])
        for i in range(len(notes) - context_length)
    ]

## 4. One-Hot Encoding

Notes are **categorical**, not ordinal — note 64 is not 'greater than' note 63 in any musical sense. Treating them as raw integers would impose a false ordering and introduce meaningless magnitude differences in the input space.

Each note index is converted to a one-hot vector of length 128. A 20-note context window becomes a `(20, 128)` matrix, then flattened to a `(2560,)` feature vector per training sample.

In [ ]:
def make_onehot(indices, total=NUM_NOTES):
    """
    Convert note indices to one-hot vectors.

    Parameters
    ----------
    indices : np.ndarray of shape (N,) or (N, context_length)
    total   : int — vocabulary size (128 MIDI notes)

    Returns
    -------
    np.ndarray — one-hot encoded, shape (..., total)
    """
    eye = np.eye(total)
    return eye[indices]


def get_X_t(D):
    """
    Build the feature matrix X and target vector t from training pairs.

    Parameters
    ----------
    D : list of (x, t) pairs from gen_input_output

    Returns
    -------
    X : np.ndarray (N, context_length * 128) — flattened one-hot contexts
    t : np.ndarray (N,) — target note indices
    """
    xs = np.array([x for x, _ in D])   # (N, context_length)
    ts = np.array([t for _, t in D])   # (N,)
    X  = make_onehot(xs).reshape(len(D), -1)  # (N, 2560)
    return X, ts

In [ ]:
# Build datasets — train on most pieces, hold out one for validation
train_files = chopin_files[:-2]
valid_files = chopin_files[-2:-1]
test_files  = chopin_files[-1:]

def load_dataset(files):
    pairs = []
    for f in files:
        notes = get_midi_notes(f)
        pairs.extend(gen_input_output(notes))
    return get_X_t(pairs)

X_train, t_train = load_dataset(train_files)
X_valid, t_valid = load_dataset(valid_files)
X_test,  t_test  = load_dataset(test_files)

print(f'Train:      {X_train.shape}  |  {len(train_files)} pieces')
print(f'Validation: {X_valid.shape}  |  {len(valid_files)} piece')
print(f'Test:       {X_test.shape}   |  {len(test_files)} piece')
print(f'\nFeature vector: {X_train.shape[1]} = {CONTEXT_LENGTH} notes × {NUM_NOTES} one-hot dims')

## 5. MLP Architecture

A 2-layer MLP implemented entirely in NumPy.

```
Input (2560)  →  Hidden (100, ReLU)  →  Output (128, Softmax)
```

**Forward pass equations:**
```
m = W1 @ x + b1        # pre-activation, hidden layer
h = ReLU(m)            # hidden activation
z = W2 @ h + b2        # pre-activation, output layer
y = softmax(z)         # probability distribution over 128 notes
```

The softmax output is numerically stabilized by subtracting the row-wise maximum before exponentiation (prevents overflow for large logits).

In [ ]:
def softmax(z):
    """
    Numerically stable softmax — row-wise for batch input.

    Subtracts the max logit per row before exponentiation to prevent overflow.

    Parameters
    ----------
    z : np.ndarray (K,) or (N, K)

    Returns
    -------
    np.ndarray — same shape as z, rows sum to 1
    """
    if z.ndim == 1:
        z = z - z.max()
        e = np.exp(z)
        return e / e.sum()
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

In [ ]:
class MLPModel:
    """
    2-layer MLP for next-note prediction.

    Architecture: input (num_features) → hidden (num_hidden, ReLU) → output (num_classes, softmax)
    """

    def __init__(self, num_features=NUM_NOTES * CONTEXT_LENGTH,
                 num_hidden=100, num_classes=NUM_NOTES):
        self.num_features = num_features
        self.num_hidden   = num_hidden
        self.num_classes  = num_classes

        # He initialization — appropriate for ReLU activations
        scale = np.sqrt(2.0 / num_features)
        self.W1 = np.random.randn(num_features, num_hidden) * scale
        self.b1 = np.zeros(num_hidden)
        self.W2 = np.random.randn(num_hidden, num_classes) * np.sqrt(2.0 / num_hidden)
        self.b2 = np.zeros(num_classes)

        # Cached forward-pass quantities (populated by do_forward_pass)
        self.X = self.m = self.h = self.z = self.y = None

## 6. Forward Pass

In [ ]:
def do_forward_pass(model, X):
    """
    Compute the forward pass for a batch of inputs.

    Caches intermediate values on the model object for use in backprop.

    Parameters
    ----------
    model : MLPModel
    X     : np.ndarray (N, num_features)

    Returns
    -------
    y : np.ndarray (N, num_classes) — predicted note probabilities
    """
    model.X = X
    model.m = X @ model.W1 + model.b1          # (N, num_hidden)
    model.h = np.maximum(0, model.m)            # ReLU
    model.z = model.h @ model.W2 + model.b2    # (N, num_classes)
    model.y = softmax(model.z)                  # (N, num_classes)
    return model.y

## 7. Music Generation — Autoregressive Sampling

The model generates music one note at a time:
1. Feed the last 20 notes through the forward pass → probability distribution over 128 notes
2. Sample the next note from that distribution
3. Append to the sequence, slide the window forward
4. Repeat

Using **sampling** rather than argmax (always picking the most likely note) introduces variation and prevents the model from immediately collapsing into repetitive loops.

In [ ]:
def generate_piece(model, seed, max_len=100):
    """
    Autoregressively generate a note sequence.

    Parameters
    ----------
    model   : MLPModel (trained or untrained)
    seed    : list of int — initial context (length >= CONTEXT_LENGTH)
    max_len : int — total number of notes to generate

    Returns
    -------
    list of int — generated note sequence
    """
    notes = list(seed)

    while len(notes) < max_len:
        context = notes[-CONTEXT_LENGTH:]
        x = make_onehot(np.array(context)).reshape(1, -1)  # (1, 2560)
        y = do_forward_pass(model, x)[0]                   # (128,)
        next_note = np.random.choice(NUM_NOTES, p=y)
        notes.append(int(next_note))

    return notes


def generate_midi(notes, outfile, tempo_ms=128):
    """
    Convert a note sequence to a MIDI file.

    Parameters
    ----------
    notes     : list of int — pitch values
    outfile   : str — output .mid path
    tempo_ms  : int — time between notes in MIDI ticks
    """
    mid = MidiFile()
    track = MidiTrack()
    mid.tracks.append(track)
    for note in notes:
        track.append(Message('note_on',  note=note, velocity=64, time=tempo_ms))
        track.append(Message('note_off', note=note, velocity=64, time=tempo_ms))
    mid.save(outfile)
    print(f'Saved: {outfile}')

## 8. Backpropagation — Vectorized

The backward pass computes gradients of the cross-entropy loss with respect to all parameters using the chain rule.

**Gradient derivations (batch notation):**
```
z_bar  = (y - t_onehot) / N          # dL/dz — softmax + CE simplifies cleanly
W2_bar = h.T @ z_bar                 # dL/dW2
b2_bar = z_bar.sum(axis=0)           # dL/db2
h_bar  = z_bar @ W2.T                # dL/dh
m_bar  = h_bar * (m > 0)             # dL/dm — ReLU gradient (0 where pre-activation <= 0)
W1_bar = X.T @ m_bar                 # dL/dW1
b1_bar = m_bar.sum(axis=0)           # dL/db1
```

All operations are fully vectorized — no Python loops over examples.

In [ ]:
def do_backward_pass(model, ts):
    """
    Compute gradients via backpropagation.

    Assumes do_forward_pass has been called for the corresponding input X,
    so intermediate values are cached on the model object.

    Parameters
    ----------
    model : MLPModel with cached forward-pass values
    ts    : np.ndarray (N,) — target note indices

    Stores dW1, db1, dW2, db2 on the model object.
    """
    N = len(ts)

    # One-hot encode targets
    t_onehot = make_onehot(ts)  # (N, 128)

    # Output layer gradient — softmax + cross-entropy derivative simplifies to (y - t) / N
    z_bar  = (model.y - t_onehot) / N          # (N, num_classes)

    model.dW2 = model.h.T @ z_bar              # (num_hidden, num_classes)
    model.db2 = z_bar.sum(axis=0)              # (num_classes,)

    # Hidden layer gradient
    h_bar  = z_bar @ model.W2.T                # (N, num_hidden)
    m_bar  = h_bar * (model.m > 0)             # ReLU: zero gradient where pre-activation <= 0

    model.dW1 = model.X.T @ m_bar              # (num_features, num_hidden)
    model.db1 = m_bar.sum(axis=0)              # (num_hidden,)

## 9. Training — Mini-Batch SGD

In [ ]:
def cross_entropy_loss(y, ts):
    """Average cross-entropy loss across a batch."""
    N = len(ts)
    return -np.log(y[np.arange(N), ts] + 1e-12).mean()


def train_sgd(model, X_train, t_train,
              alpha=0.1, n_epochs=30, batch_size=100,
              X_valid=None, t_valid=None, plot=True):
    """
    Train the MLP with mini-batch SGD.

    Parameters
    ----------
    model      : MLPModel
    X_train    : np.ndarray (N, num_features)
    t_train    : np.ndarray (N,) — target indices
    alpha      : float — learning rate
    n_epochs   : int
    batch_size : int
    X_valid    : optional validation features
    t_valid    : optional validation targets
    plot       : bool — plot loss curves after training

    Returns
    -------
    train_losses, val_losses : lists of per-epoch losses
    """
    N = X_train.shape[0]
    train_losses, val_losses = [], []

    for epoch in range(n_epochs):
        # Shuffle training data
        perm = np.random.permutation(N)
        X_train, t_train = X_train[perm], t_train[perm]

        # Mini-batch updates
        for start in range(0, N, batch_size):
            X_batch = X_train[start : start + batch_size]
            t_batch = t_train[start : start + batch_size]

            do_forward_pass(model, X_batch)
            do_backward_pass(model, t_batch)

            model.W1 -= alpha * model.dW1
            model.b1 -= alpha * model.db1
            model.W2 -= alpha * model.dW2
            model.b2 -= alpha * model.db2

        # Record epoch loss
        y_train = do_forward_pass(model, X_train)
        train_losses.append(cross_entropy_loss(y_train, t_train))

        if X_valid is not None:
            y_valid = do_forward_pass(model, X_valid)
            val_losses.append(cross_entropy_loss(y_valid, t_valid))

        if (epoch + 1) % 5 == 0:
            v = f'{val_losses[-1]:.4f}' if val_losses else 'N/A'
            print(f'Epoch {epoch+1:3d}/{n_epochs}  |  train: {train_losses[-1]:.4f}  |  val: {v}')

    if plot:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(train_losses, label='Train loss', color='steelblue', linewidth=2)
        if val_losses:
            ax.plot(val_losses, label='Validation loss', color='coral', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Cross-entropy loss')
        ax.set_title('Training Curve — MLP Music Generation Model', fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig('images/training_curve.png', dpi=150, bbox_inches='tight')
        plt.show()

    return train_losses, val_losses

## 10. Sanity Check — Overfit a Single Batch

Before training on the full dataset, verify the implementation by confirming the model can overfit 100 examples to near-zero loss within 500 iterations. If it can't, there's a bug in either the forward or backward pass.

In [ ]:
X_small, t_small = X_train[:100], t_train[:100]
model_check = MLPModel()
train_sgd(model_check, X_small, t_small, alpha=0.2, batch_size=100, n_epochs=500)
print('If training loss is near 0, the implementation is correct.')

## 11. Full Training on Chopin Dataset

In [ ]:
model = MLPModel()
train_losses, val_losses = train_sgd(
    model,
    X_train, t_train,
    alpha=0.1,
    batch_size=100,
    n_epochs=30,
    X_valid=X_valid,
    t_valid=t_valid,
)

## 12. Generate Original Compositions

Use the trained model to generate new pieces. The seed is the opening 20 notes of a Chopin piece — the model then continues it in a learned style.

In [ ]:
# Load seed from a test piece
seed_notes = get_midi_notes(test_files[0])[:CONTEXT_LENGTH]

# Generate 200-note composition
generated_notes = generate_piece(model, seed_notes, max_len=200)
generate_midi(generated_notes, 'generated_composition.mid')

print(f'Seed (first {CONTEXT_LENGTH} notes): {seed_notes}')
print(f'Generated ({len(generated_notes)} notes): {generated_notes[:40]}...')

In [ ]:
# Visualize the generated note sequence as a piano roll
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Seed
axes[0].bar(range(len(seed_notes)), seed_notes, color='steelblue', width=0.8)
axes[0].set_title('Seed Sequence (first 20 notes)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('MIDI note (pitch)')
axes[0].set_ylim(0, 128)
axes[0].axhline(60, color='gray', linestyle='--', alpha=0.5, linewidth=0.8, label='Middle C (60)')
axes[0].legend(fontsize=9)

# Generated
axes[1].bar(range(len(generated_notes)), generated_notes, color='coral', width=0.8, alpha=0.85)
axes[1].set_title('Generated Composition (200 notes)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Note position in sequence')
axes[1].set_ylabel('MIDI note (pitch)')
axes[1].set_ylim(0, 128)
axes[1].axhline(60, color='gray', linestyle='--', alpha=0.5, linewidth=0.8, label='Middle C (60)')
axes[1].legend(fontsize=9)

plt.suptitle('Piano Roll — Seed vs. Generated Composition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/piano_roll.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Note Distribution Analysis

Comparing the pitch distribution of the generated composition against the training data confirms the model has learned the statistical structure of Chopin's note choices — not just random sampling.

In [ ]:
all_train_notes = []
for f in train_files:
    all_train_notes.extend(get_midi_notes(f))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(all_train_notes, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Training Data — Note Distribution', fontsize=11, fontweight='bold')
axes[0].set_xlabel('MIDI pitch')
axes[0].set_ylabel('Frequency')

axes[1].hist(generated_notes, bins=50, color='coral', alpha=0.8, edgecolor='white')
axes[1].set_title('Generated Composition — Note Distribution', fontsize=11, fontweight='bold')
axes[1].set_xlabel('MIDI pitch')
axes[1].set_ylabel('Frequency')

plt.suptitle('Pitch Distribution: Training vs. Generated', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/note_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Architecture Limitations — MLP vs. RNNs vs. Transformers

The MLP generates musically plausible short sequences but degrades over longer outputs. Understanding *why* informed the architecture recommendations delivered to the client.

### Why the MLP collapses into repetition

The model's only input is a **fixed-length window of 20 notes**. Once the generated sequence enters a pattern that's locally likely, the fixed context window keeps feeding that same pattern back in — the model has no mechanism to 'remember' or 'escape' from it. It doesn't model global musical structure, only local note-to-note transitions.

### Architectural comparison

| Architecture | Context | Long-range dependency | Generation quality |
|---|---|---|---|
| **MLP (this project)** | Fixed window (20 notes) | None | Short sequences only |
| **RNN / LSTM** | Unbounded (hidden state) | Medium (vanishing gradient) | Better — can track key/tempo |
| **Transformer** | Full sequence (attention) | Strong | State-of-the-art — e.g. MuseNet, MusicLM |

### Recommendation

For production music generation, the MLP serves as a validated baseline. The next iteration should replace the fixed-window MLP with an LSTM or Transformer decoder, which can maintain harmonic and rhythmic consistency across longer sequences.

## 15. Summary

| Component | Detail |
|---|---|
| Training data | Chopin piano pieces (simplified MIDI) |
| Context window | 20 notes |
| Feature vector | 2,560 dims (20 × 128 one-hot) |
| Architecture | MLP: 2560 → 100 (ReLU) → 128 (Softmax) |
| Training | Mini-batch SGD, 30 epochs, lr=0.1 |
| Backprop | Fully vectorized NumPy — no frameworks |
| Output | .mid files playable in any MIDI player |

**Delivered:**
- Trained model generating Chopin-style compositions
- Autoregressive generation pipeline
- Architecture comparison (MLP → RNN → Transformer) with documented tradeoffs
- Full reproducible pipeline — no ML framework dependencies (NumPy only)